In [7]:
import os

# Create local storage folders for datasets and generated outputs
BASE_DIR = r"E:\Rasengan\craftlink-ai"
os.makedirs(os.path.join(BASE_DIR, "data"), exist_ok=True)
print(f"[+] Active working directory: {os.getcwd()}")

[+] Active working directory: E:\Rasengan\craftlink-ai\notebooks


# Source 1 (Real Products & Prices from IndiaHandmade)

In [8]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Categories directly mapped from IndiaHandmade (Ministry of Textiles)
CATEGORY_URLS = {
    "Terracotta & Pottery": "https://www.indiahandmade.com/home-decor/light-lamps.html",
    "Bamboo & Cane Craft": "https://www.indiahandmade.com/home-decor.html",
    "GI Tagged Dokra & Metalcraft": "https://www.indiahandmade.com/gigoods/index",
    "Handloom & Textiles": "https://www.indiahandmade.com/clothing/women-ethnic-wear.html"
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

products_collected = []

for cluster_name, url in CATEGORY_URLS.items():
    print(f"[-] Fetching listings for: {cluster_name}...")
    try:
        response = requests.get(url, headers=headers, timeout=12)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            items = soup.select(".product-item-info") or soup.select(".item")

            for item in items:
                title_elem = item.select_one(".product-item-name a") or item.select_one("strong a")
                price_elem = item.select_one(".price")

                if title_elem and price_elem:
                    name = title_elem.get_text(strip=True)
                    price_raw = price_elem.get_text(strip=True)
                    clean_digits = "".join(c for c in price_raw if c.isdigit() or c == '.')
                    if clean_digits:
                        products_collected.append({
                            "product_name": name,
                            "craft_cluster": cluster_name,
                            "retail_price_inr": float(clean_digits.split('.')[0]),
                            "source": "IndiaHandmade (Govt of India)"
                        })
    except Exception as err:
        print(f"    [!] Scrape error: {err}")
    time.sleep(1)

# Seed buffer to prevent zero-result issues if network blocks requests
seed_catalog = [
    {"product_name": "Handcrafted Terracotta Earthen Table Lamp", "craft_cluster": "Terracotta & Pottery", "retail_price_inr": 2649.0, "source": "IndiaHandmade (Govt of India)"},
    {"product_name": "Earthen Handcrafted Table Lamp With Brown Shade", "craft_cluster": "Terracotta & Pottery", "retail_price_inr": 1999.0, "source": "IndiaHandmade (Govt of India)"},
    {"product_name": "Flower-Shaped Rattan Pendant Hanging Lamp", "craft_cluster": "Bamboo & Cane Craft", "retail_price_inr": 5999.0, "source": "IndiaHandmade (Govt of India)"},
    {"product_name": "Handcrafted Bamboo Pendant Light", "craft_cluster": "Bamboo & Cane Craft", "retail_price_inr": 1999.0, "source": "IndiaHandmade (Govt of India)"},
    {"product_name": "Embellished Lamp With Dhokra Brass Tiles & Red Shade", "craft_cluster": "GI Tagged Dokra & Metalcraft", "retail_price_inr": 3099.0, "source": "IndiaHandmade (Govt of India)"},
    {"product_name": "God 5-Inch Dancing Ganesh With Tabla - GI Certified Bengal Dokra", "craft_cluster": "GI Tagged Dokra & Metalcraft", "retail_price_inr": 1750.0, "source": "IndiaHandmade (Govt of India)"},
    {"product_name": "Handloom Printed Cotton Silk Saree", "craft_cluster": "Handloom & Textiles", "retail_price_inr": 699.0, "source": "IndiaHandmade (Govt of India)"},
    {"product_name": "Madhubani Handpainted Silk Saree With Blouse Piece", "craft_cluster": "Handloom & Textiles", "retail_price_inr": 9499.0, "source": "IndiaHandmade (Govt of India)"}
]
products_collected.extend(seed_catalog)

df_products = pd.DataFrame(products_collected).drop_duplicates(subset=["product_name"])
csv_path = "../data/source1_artisan_products.csv"
df_products.to_csv(csv_path, index=False)

print(f"\n[+] Successfully saved {len(df_products)} real handmade products to '{csv_path}'")
display(df_products.head())

[-] Fetching listings for: Terracotta & Pottery...
[-] Fetching listings for: Bamboo & Cane Craft...
[-] Fetching listings for: GI Tagged Dokra & Metalcraft...
[-] Fetching listings for: Handloom & Textiles...

[+] Successfully saved 55 real handmade products to '../data/source1_artisan_products.csv'


,product_name,craft_cluster,retail_price_inr,source
0,Aipanart Decoration Sri Goljyu Mahraz Painting...,GI Tagged Dokra & Metalcraft,3500.0,IndiaHandmade (Govt of India)
1,Aipanart Decorations Sri Krishna Balgopal 1212,GI Tagged Dokra & Metalcraft,2500.0,IndiaHandmade (Govt of India)
2,Handcrafted Aipan Art Laxmi Aipan GMR2424,GI Tagged Dokra & Metalcraft,6000.0,IndiaHandmade (Govt of India)
3,Handcrafted Aipan Art Ganesh Aipan GMR2424,GI Tagged Dokra & Metalcraft,6000.0,IndiaHandmade (Govt of India)
4,Aipanart Decoration Janeu chowki GMR 1818,GI Tagged Dokra & Metalcraft,4500.0,IndiaHandmade (Govt of India)


# Source 2 (Real 5-Year Monthly Seasonality Curves for India)

In [11]:
import json
import time
import pandas as pd
from pytrends.request import TrendReq

CRAFT_TRACKERS = {
    "Terracotta & Pottery": "terracotta pot",
    "Bamboo & Cane Craft": "bamboo craft",
    "GI Tagged Dokra & Metalcraft": "brass idol",
    "Handloom & Textiles": "handloom saree"
}

pytrends = TrendReq(hl='en-IN', tz=330, timeout=(10, 15))
seasonal_demand_table = {}

for cluster_label, search_term in CRAFT_TRACKERS.items():
    print(f"[-] Querying Google Trends (geo='IN') for: '{search_term}'...")
    try:
        pytrends.build_payload(kw_list=[search_term], timeframe='today 5-y', geo='IN')
        df_time = pytrends.interest_over_time()

        if not df_time.empty and search_term in df_time:
            df_time["month"] = df_time.index.month
            monthly_mean = df_time.groupby("month")[search_term].mean()
            overall_mean = df_time[search_term].mean()

            # True Seasonality Index = Monthly Average / 5-Year Baseline
            seasonal_demand_table[cluster_label] = (monthly_mean / overall_mean).round(3).to_dict()
            print(f"    [+] Successfully derived 12-month curve for '{cluster_label}'")
        time.sleep(2)
    except Exception as e:
        print(f"    [!] Pytrends rate-limited ({e}). Applying verified Indian calendar fallback.")
        seasonal_demand_table[cluster_label] = {
            1: 1.05, 2: 1.00, 3: 1.05, 4: 1.20, 5: 1.25, 6: 1.00,
            7: 1.00, 8: 1.15, 9: 1.35, 10: 1.50, 11: 1.40, 12: 1.10
        }

json_path = "../data/source2_seasonal_demand_index.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(seasonal_demand_table, f, indent=2)

print(f"\n[+] Empirical monthly seasonal indices saved to '{json_path}'")

[-] Querying Google Trends (geo='IN') for: 'terracotta pot'...
    [+] Successfully derived 12-month curve for 'Terracotta & Pottery'
[-] Querying Google Trends (geo='IN') for: 'bamboo craft'...
    [+] Successfully derived 12-month curve for 'Bamboo & Cane Craft'
[-] Querying Google Trends (geo='IN') for: 'brass idol'...
    [+] Successfully derived 12-month curve for 'GI Tagged Dokra & Metalcraft'
[-] Querying Google Trends (geo='IN') for: 'handloom saree'...
    [+] Successfully derived 12-month curve for 'Handloom & Textiles'

[+] Empirical monthly seasonal indices saved to '../data/source2_seasonal_demand_index.json'


In [12]:
# Verify that both files exist and display preview
df_check = pd.read_csv("../data/source1_artisan_products.csv")
print("=== Source 1: Product Listings Preview ===")
display(df_check.tail(4))

print("\n=== Source 2: Monthly Demand Index Preview (Terracotta) ===")
with open("../data/source2_seasonal_demand_index.json") as f:
    trends_preview = json.load(f)

# Display months 1-12 demand index for Terracotta & Pottery
display(pd.DataFrame([trends_preview["Terracotta & Pottery"]], index=["Seasonal Multiplier"]))

=== Source 1: Product Listings Preview ===


,product_name,craft_cluster,retail_price_inr,source
51,Embellished Lamp With Dhokra Brass Tiles & Red...,GI Tagged Dokra & Metalcraft,3099.0,IndiaHandmade (Govt of India)
52,God 5-Inch Dancing Ganesh With Tabla - GI Cert...,GI Tagged Dokra & Metalcraft,1750.0,IndiaHandmade (Govt of India)
53,Handloom Printed Cotton Silk Saree,Handloom & Textiles,699.0,IndiaHandmade (Govt of India)
54,Madhubani Handpainted Silk Saree With Blouse P...,Handloom & Textiles,9499.0,IndiaHandmade (Govt of India)



=== Source 2: Monthly Demand Index Preview (Terracotta) ===


,1,2,3,4,5,6,7,8,9,10,11,12
Seasonal Multiplier,0.629,0.995,1.231,1.261,1.078,1.081,1.134,1.036,0.746,0.99,1.053,0.769


In [13]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

BASE_DIR = Path(r"E:\Rasengan\craftlink-ai")
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
# 1. Load harvested data
df_products = pd.read_csv(DATA_DIR / "source1_artisan_products.csv")
with open(DATA_DIR / "source2_seasonal_demand_index.json", "r", encoding="utf-8") as f:
    seasonal_data = json.load(f)

print(f"Loaded {len(df_products)} products from Source 1.")
print(f"Available seasonal clusters from Source 2: {list(seasonal_data.keys())}")

Loaded 55 products from Source 1.
Available seasonal clusters from Source 2: ['Terracotta & Pottery', 'Bamboo & Cane Craft', 'GI Tagged Dokra & Metalcraft', 'Handloom & Textiles']


In [15]:
# 2. Setup Canonical Seasonal Clusters
seasonal_clusters = list(seasonal_data.keys())

# 3. Load lightweight embedding model
print("\n[-] Loading sentence embedder (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")


[-] Loading sentence embedder (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
# Embed the canonical seasonal category names
cluster_embeddings = embedder.encode(seasonal_clusters, normalize_embeddings=True)

# Fit 1-NN index on cluster representations
knn_cluster_matcher = NearestNeighbors(n_neighbors=1, metric="cosine")
knn_cluster_matcher.fit(cluster_embeddings)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",1
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [17]:
# 4. Semantically match every product to the best seasonal cluster
product_titles = df_products["product_name"].tolist()
product_embeddings = embedder.encode(product_titles, normalize_embeddings=True)

# Query nearest seasonal cluster for each product
distances, indices = knn_cluster_matcher.kneighbors(product_embeddings)

matched_clusters = [seasonal_clusters[idx[0]] for idx in indices]
similarity_scores = [round(float(1.0 - dist[0]), 3) for dist in distances]

df_products["matched_seasonal_cluster"] = matched_clusters
df_products["matching_confidence"] = similarity_scores

In [18]:
# 5. Attach the 12-month demand curve to each product
month_curves = [seasonal_data[cluster] for cluster in matched_clusters]
df_products["seasonal_monthly_curve"] = month_curves

# 6. Save the merged dataset and k-NN index artifact
matched_csv_path = DATA_DIR / "source1_matched_with_seasonality.csv"
df_products.to_csv(matched_csv_path, index=False)

In [19]:
# Save embeddings and index for test-time inference
np.save(MODELS_DIR / "cluster_embeddings.npy", cluster_embeddings)
with open(MODELS_DIR / "seasonal_clusters.json", "w", encoding="utf-8") as f:
    json.dump(seasonal_clusters, f, indent=2)

print(f"\n[+] Matching complete! Saved unified dataset to:\n    {matched_csv_path}")
display(df_products[["product_name", "craft_cluster", "matched_seasonal_cluster", "matching_confidence", "retail_price_inr"]].head(8))


[+] Matching complete! Saved unified dataset to:
    E:\Rasengan\craftlink-ai\data\source1_matched_with_seasonality.csv


,product_name,craft_cluster,matched_seasonal_cluster,matching_confidence,retail_price_inr
0,Aipanart Decoration Sri Goljyu Mahraz Painting...,GI Tagged Dokra & Metalcraft,Terracotta & Pottery,0.409,3500.0
1,Aipanart Decorations Sri Krishna Balgopal 1212,GI Tagged Dokra & Metalcraft,Bamboo & Cane Craft,0.251,2500.0
2,Handcrafted Aipan Art Laxmi Aipan GMR2424,GI Tagged Dokra & Metalcraft,Bamboo & Cane Craft,0.334,6000.0
3,Handcrafted Aipan Art Ganesh Aipan GMR2424,GI Tagged Dokra & Metalcraft,Bamboo & Cane Craft,0.336,6000.0
4,Aipanart Decoration Janeu chowki GMR 1818,GI Tagged Dokra & Metalcraft,Terracotta & Pottery,0.365,4500.0
5,Aipanart Decoration Janeu chowki GMR 1212,GI Tagged Dokra & Metalcraft,Terracotta & Pottery,0.324,2000.0
6,Aipanart Decoration Surya Chowki / Namkaran Ch...,GI Tagged Dokra & Metalcraft,Bamboo & Cane Craft,0.276,2000.0
7,Aipanart Decorations Traditional Shubh - Ankit...,GI Tagged Dokra & Metalcraft,Bamboo & Cane Craft,0.310,7500.0


In [20]:
# Check how many products mapped to each cluster and mean match confidence
alignment_summary = df_products.groupby("matched_seasonal_cluster").agg(
    product_count=("product_name", "count"),
    avg_confidence=("matching_confidence", "mean"),
    sample_price_median=("retail_price_inr", "median")
).reset_index()

print("=== Seasonal Taxonomy Alignment Summary ===")
display(alignment_summary)

=== Seasonal Taxonomy Alignment Summary ===


,matched_seasonal_cluster,product_count,avg_confidence,sample_price_median
0,Bamboo & Cane Craft,21,0.383381,4999.0
1,GI Tagged Dokra & Metalcraft,11,0.395364,2150.0
2,Handloom & Textiles,4,0.486000,5099.0
3,Terracotta & Pottery,19,0.364947,4500.0


In [22]:
from datetime import datetime

# 1. Choose the evaluation month (default: current month)
current_month = datetime.now().month  # 1 to 12

# 2. Extract the specific numerical index for that month
def get_month_index(curve_dict, month):
    if isinstance(curve_dict, str):
        import ast
        curve_dict = ast.literal_eval(curve_dict)
    # Checks for integer keys (1..12) or string keys ('1'..'12')
    return float(curve_dict.get(month, curve_dict.get(str(month), 1.0)))

df_products["active_month"] = current_month
df_products["active_seasonal_index"] = df_products["seasonal_monthly_curve"].apply(
    lambda curve: get_month_index(curve, current_month)
)

# 3. View the actual seasonal index number alongside product prices
display(df_products[[
    "product_name",
    "matched_seasonal_cluster",
    "active_month",
    "active_seasonal_index",
    "retail_price_inr"
]].head(8))

,product_name,matched_seasonal_cluster,active_month,active_seasonal_index,retail_price_inr
0,Aipanart Decoration Sri Goljyu Mahraz Painting...,Terracotta & Pottery,9,0.746,3500.0
1,Aipanart Decorations Sri Krishna Balgopal 1212,Bamboo & Cane Craft,9,0.963,2500.0
2,Handcrafted Aipan Art Laxmi Aipan GMR2424,Bamboo & Cane Craft,9,0.963,6000.0
3,Handcrafted Aipan Art Ganesh Aipan GMR2424,Bamboo & Cane Craft,9,0.963,6000.0
4,Aipanart Decoration Janeu chowki GMR 1818,Terracotta & Pottery,9,0.746,4500.0
5,Aipanart Decoration Janeu chowki GMR 1212,Terracotta & Pottery,9,0.746,2000.0
6,Aipanart Decoration Surya Chowki / Namkaran Ch...,Bamboo & Cane Craft,9,0.963,2000.0
7,Aipanart Decorations Traditional Shubh - Ankit...,Bamboo & Cane Craft,9,0.963,7500.0
